In [ ]:
# %% [markdown]
# # M5 Decagon Ensemble: Topology & Signal Audit
# **Aim:** Verify the mathematical integrity of the 10 Graph Views before training.
# **Research Focus:** Density, Spectral Gap, and Degree Distribution.

# %%
import torch
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import yaml
from torch_geometric.utils import to_networkx, degree
from scipy.stats import entropy

# %%
# 1. Configuration & Loading
with open('../configs/data_config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

def load_view(view_name):
    path = f"../data/graphs/{cfg['graph_views'][view_name]}"
    return torch.load(path)

# %%
# 2. Spectral Density & Sparsity Audit
# We check 'Behavioral' (Correlation) and 'Economic' views specifically.
views_to_audit = ['behavioral', 'economic', 'temporal_sync']

for view in views_to_audit:
    edge_index = load_view(view)
    num_nodes = 30490
    num_edges = edge_index.shape[1]
    
    # Calculate Density: Standard GNNs fail if density > 0.01 (Oversmoothing risk)
    density = num_edges / (num_nodes * (num_nodes - 1))
    
    # Calculate Degree Distribution
    deg = degree(edge_index[0], num_nodes)
    avg_deg = deg.mean().item()
    max_deg = deg.max().item()
    
    # Degree Entropy: High entropy = diverse connections; Low = bottlenecked
    deg_hist = torch.histc(deg, bins=50, min=0, max=max_deg)
    deg_entropy = entropy(deg_hist.numpy() + 1e-9)

    print(f"--- Audit: {view.upper()} ---")
    print(f"Density: {density:.8f} | Avg Degree: {avg_deg:.2f} | Max Degree: {max_deg}")
    print(f"Degree Entropy: {deg_entropy:.4f}")
    if density > 0.005:
        print("⚠️ WARNING: High density detected. Increase threshold in GraphBuilder.")
    print("\n")

# %%
# 3. Community Detection (Modularity Audit)
# We use the Louvain method to see if the GNN will naturally find 'Baskets'
view_to_viz = 'behavioral'
edge_index = load_view(view_to_viz)

# Sample a Store (e.g., CA_1) to keep the graph size manageable for viz
# CA_1 items are typically the first ~3000 nodes
sample_mask = (edge_index[0] < 3049) & (edge_index[1] < 3049)
G = to_networkx(torch.cat([edge_index[:, sample_mask]]), to_undirected=True)

# Visualize Connectivity Patterns
plt.figure(figsize=(15, 10))
pos = nx.spring_layout(G, k=0.15, iterations=20)
nx.draw_networkx_nodes(G, pos, node_size=5, node_color='blue', alpha=0.3)
nx.draw_networkx_edges(G, pos, alpha=0.1, edge_color='gray')
plt.title(f"Latent Manifold Visualization: {view_to_viz} (Store: CA_1)")
plt.axis('off')
plt.show()

# %%
# 4. Message Passing 'Horizon' Check
# Calculates the diameter of the graph to see how many GNN layers are needed.
if nx.is_connected(G):
    diameter = nx.diameter(G)
    print(f"Graph Diameter (CA_1): {diameter}")
    print(f"Recommended GNN Layers: {min(diameter, 5)}")
else:
    print("Graph is disconnected. Experts will rely on local neighborhood info.")